In [ ]:

# [0. 필요한 라이브러리 설치 및 임포트]
!pip install noisereduce -q  # 노이즈 제거 라이브러리 설치

import os
import random
import shutil
import numpy as np
import librosa
import soundfile as sf
import cv2
import matplotlib.pyplot as plt
import noisereduce as nr  # 전처리 노트북의 노이즈 제거 라이브러리 추가
from google.colab import drive
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models


# [1. 경로 및 설정 값]

DATASET_PATH = '/content/drive/MyDrive/오디오파일wav'
classes = ['0_indoor_alarms', '1_outdoor_warnings', '2_emergency_alarms']
SAMPLE_RATE = 22050  # 표준 샘플링 레이트 통일


# [1.5. 업로드된 노트북의 전처리(Preprocessing) 파이프라인 정의]

def reduce_noise(y, sr=SAMPLE_RATE):
    """스펙트럴 게이팅 기반 노이즈 제거. 잡음 구간을 따로 지정하지 않아도
    오디오 전체에서 잡음 프로파일을 통계적으로 추정해 제거합니다."""
    try:
        y_denoised = nr.reduce_noise(y=y, sr=sr, stationary=False)
    except Exception as e:
        print(f"  [경고] 노이즈 제거 실패, 원본 사용: {e}")
        y_denoised = y
    return y_denoised

def trim_silence(y, top_db=30):
    """앞뒤에 붙은 무음(또는 거의 무음인) 구간을 제거합니다.
    top_db가 낮을수록 더 작은 소리도 '소리 있음'으로 인식해 덜 자릅니다."""
    y_trimmed, _ = librosa.effects.trim(y, top_db=top_db)
    if len(y_trimmed) == 0:
        return y  # 전부 잘려나가는 경우를 방지하는 안전장치
    return y_trimmed

def peak_normalize(y, target_peak=0.95):
    """피크 기준으로 음량을 정규화해서 파일 간 음량 차이를 통일합니다."""
    max_val = np.max(np.abs(y))
    if max_val > 0:
        y = y / max_val * target_peak
    return y

def preprocess_audio(y, sr=SAMPLE_RATE):
    """전처리 파이프라인: 노이즈 제거 -> 무음 트리밍 -> 정규화"""
    y = reduce_noise(y, sr)
    y = trim_silence(y)
    y = peak_normalize(y)
    return y

# [2. 오디오 개별 증강 함수 정의]

def time_shift(y, sr=SAMPLE_RATE, shift_max=0.3):
    shift_amt = int(sr * random.uniform(-shift_max, shift_max))
    return np.roll(y, shift_amt)

def pitch_shift(y, sr=SAMPLE_RATE, n_steps_range=(-3, 3)):
    n_steps = random.uniform(*n_steps_range)
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)

def time_stretch(y, rate_range=(0.85, 1.15)):
    rate = random.uniform(*rate_range)
    return librosa.effects.time_stretch(y, rate=rate)

def add_noise(y, noise_factor_range=(0.002, 0.01)):
    noise_factor = random.uniform(*noise_factor_range)
    noise = np.random.randn(len(y))
    return y + noise_factor * noise

def change_volume(y, gain_range=(0.6, 1.4)):
    gain = random.uniform(*gain_range)
    y_new = y * gain
    max_val = np.max(np.abs(y_new))
    if max_val > 1.0:
        y_new = y_new / max_val
    return y_new

def time_mask(y, sr=SAMPLE_RATE, mask_max_ratio=0.1):
    y_new = y.copy()
    mask_len = int(len(y) * random.uniform(0.02, mask_max_ratio))
    if mask_len <= 0 or len(y) <= mask_len:
        return y_new
    start = random.randint(0, len(y) - mask_len)
    y_new[start:start + mask_len] = 0.0
    return y_new

# 리스트 형태로 관리하여 반복문으로 처리할 수 있도록 구성
AUGMENTATIONS = [
    time_shift,
    pitch_shift,
    time_stretch,
    add_noise,
    change_volume,
    time_mask
]


# [3. 멜-스펙트로그램 추출 및 통합 증강 함수 (전처리 파이프라인 연동)]

def extract_mel_spectrogram(file_path, augment=False):
    try:
        # 파일 로드 (sr=22050, 모노톤 적용)
        y, sr = librosa.load(file_path, sr=SAMPLE_RATE, mono=True, duration=3.0)

        # [수정] 파일 로드 직후 노트북의 전처리 파이프라인(노이즈 제거, 트리밍, 정규화) 적용
        y = preprocess_audio(y, sr=sr)
    except Exception as e:
        print(f"  [에러] {os.path.basename(file_path)} 로드 및 전처리 실패: {e}")
        return []

    features = []

    # 1. 전처리된 원본 데이터로 멜-스펙트로그램 생성
    mel = librosa.feature.melspectrogram(y=y, sr=sr)
    features.append(cv2.resize(librosa.power_to_db(mel, ref=np.max), (64, 64)))

    # 2. 증강 옵션이 켜진 경우 (훈련 데이터셋 구축 시에만 실행)
    if augment:
        for aug_func in AUGMENTATIONS:
            try:
                y_aug = aug_func(y)
                # 증강 기법 적용 후 클리핑 방지를 위해 다시 피크 정규화 수행
                y_aug = peak_normalize(y_aug)

                mel_aug = librosa.feature.melspectrogram(y=y_aug, sr=sr)
                features.append(cv2.resize(librosa.power_to_db(mel_aug, ref=np.max), (64, 64)))
            except Exception as e:
                # 특정 증강 기법 실패 시 스킵하고 다음 증강 진행
                continue

    return features


# [4. 데이터셋 파일 경로 수집]

print("\n드라이브에서 파일 목록을 수집하는 중...")
file_paths = []
file_labels = []

for idx, class_name in enumerate(classes):
    class_dir = os.path.join(DATASET_PATH, class_name)
    if not os.path.exists(class_dir):
        print(f"[경고] {class_name} 폴더를 찾을 수 없습니다. 경로를 확인하세요.")
        continue

    for file in os.listdir(class_dir):
        if file.lower().endswith('.wav'):
            file_paths.append(os.path.join(class_dir, file))
            file_labels.append(idx)

print(f"총 발견된 WAV 원본 파일 수: {len(file_paths)}개")


# [5. 데이터 누수 방지를 위한 Train / Test 분할 (8:2)]

X_train_paths, X_test_paths, y_train_paths, y_test_paths = train_test_split(
    file_paths, file_labels, test_size=0.2, random_state=42, stratify=file_labels
)


# [6. 특징 추출 및 데이터셋 빌드 (훈련셋은 1개 파일당 총 7개로 증강)]

print("\n 멜-스펙트로그램 특징 추출 및 6종 복합 증강 진행 중")
X_train, y_train = [], []
X_test, y_test = [], []

# 훈련 데이터셋 구축 (augment=True로 6종 기법 전원 적용 -> 원본 포함 총 7배 증가)
for path, label in zip(X_train_paths, y_train_paths):
    feats = extract_mel_spectrogram(path, augment=True)
    for f in feats:
        X_train.append(f)
        y_train.append(label)

# 테스트 데이터셋 구축 (순수 원본 데이터로만 평가해야 하므로 augment=False)
for path, label in zip(X_test_paths, y_test_paths):
    feats = extract_mel_spectrogram(path, augment=False)
    for f in feats:
        X_test.append(f)
        y_test.append(label)


# [7. CNN 입력을 위한 차원 조정 및 배열 변환]

X_train = np.array(X_train)[..., np.newaxis] # (데이터개수, 64, 64, 1)
y_train = np.array(y_train)
X_test = np.array(X_test)[..., np.newaxis]
y_test = np.array(y_test)

print(f"\n 최종 훈련 데이터 형태 (X_train): {X_train.shape}, 레이블 개수: {len(y_train)}")
print(f" 최종 테스트 데이터 형태 (X_test): {X_test.shape}, 레이블 개수: {len(y_test)}")


# [8. CNN 모델 설계 및 컴파일]

model = models.Sequential([
    layers.Input(shape=(64, 64, 1)),
    layers.Rescaling(1.0/80.0, offset=1.0), # -80~0 범위를 0~1 범위로 가공

    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


# [9. 모델 학습 수행]

print("\n 모델 학습 시작")
history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.1
)


# [10. 최종 테스트셋 성능 검증]

print("\n테스트 데이터로 최종 성능 평가")
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=2)
print(f"\n최종 테스트 정확도 (Test Accuracy): {test_acc * 100:.2f}%")


# [11. 학습 추이 그래프 시각화]

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy Evolution')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss Evolution')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()


# [12. 모델 및 가중치 결과물 저장 및 드라이브 백업]

model.save('my_audio_model.keras')
model.save_weights('my_audio_weights.weights.h5')

!cp my_audio_model.keras /content/drive/MyDrive/
!cp my_audio_weights.weights.h5 /content/drive/MyDrive/

print("모델 및 파일이 드라이브에 안전하게 백업되었습니다.")


드라이브에서 파일 목록을 수집하는 중...
총 발견된 WAV 원본 파일 수: 90개

 멜-스펙트로그램 특징 추출 및 6종 복합 증강 진행 중


/tmp/ipykernel_764/2846418526.py:112: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(file_path, sr=SAMPLE_RATE, mono=True, duration=3.0)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)



 최종 훈련 데이터 형태 (X_train): (504, 64, 64, 1), 레이블 개수: 504
 최종 테스트 데이터 형태 (X_test): (18, 64, 64, 1), 레이블 개수: 18


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_2 (Rescaling)         │ (None, 64, 64, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 64, 64, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 32, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 16, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │     1,048,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,141,763 (4.36 MB)

 Trainable params: 1,141,763 (4.36 MB)

 Non-trainable params: 0 (0.00 B)


 모델 학습 시작
Epoch 1/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 6s 206ms/step - accuracy: 0.3863 - loss: 1.0896 - val_accuracy: 0.7647 - val_loss: 1.0022
Epoch 2/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.6291 - loss: 0.8920 - val_accuracy: 0.4314 - val_loss: 0.9766
Epoch 3/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7506 - loss: 0.5750 - val_accuracy: 0.4902 - val_loss: 1.2685
Epoch 4/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.8896 - loss: 0.3230 - val_accuracy: 0.4902 - val_loss: 1.5330
Epoch 5/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9426 - loss: 0.1680 - val_accuracy: 0.4510 - val_loss: 2.9817
Epoch 6/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9713 - loss: 0.1059 - val_accuracy: 0.4706 - val_loss: 2.1327
Epoch 7/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9823 - loss: 0.0585 - val_accuracy: 0.4902 - val_loss: 2.6723
Epoch 8/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9956 - loss: 0.0290 - val_accuracy